# Puzzle

### This Week’s Fiddler

From Alan Pierrot comes a great way to “cap” off your week:

Three game show contestants must work together to win a prize. They are shown a large bag that’s initially empty, and then they see three red hats and two white hats placed into the bag. At this point, the contestants are blindfolded. Each contestant then picks a hat at random from the bag and places it on their head.

One at a time, their blindfolds are removed and they can see the hats on the others’ heads—but not the hat on their own head. If they can, with absolute certainty, identify the color of the hat on their own head, then the game is over and all three contestants win a prize! Otherwise, they skip their turn, at which point the next contestant has their blindfold removed and the process continues. If all three contestants skip their turn, then no prize is won. While blindfolded, contestants can still hear the decisions of other contestants and can identify who said what.

Will the contestants always win the prize? If so, why? If not, why not?

### This Week’s Extra Credit

Now, there are four game show contestants who must work together. They are shown a large bag that’s initially empty, and then they see R red hats, W white hats, and B blue hats placed into the bag, such that R, W, and B are each less than or equal to 4. Otherwise, the game is played as before.

For some triples (R, W, B), the contestants can always win. Among these, what is the greatest value of R + W + B?

# Fiddler Solution.

Let's start by enumerating all the possible hat assignments for the 3 contestants (C,D,E), assuming that blindfolds are removed in the order C,D,E.

### All possibilities
|C|D|E|Status|
|-|-|-|-|
|r|r|r||
|r|r|w||
|r|w|r||
|r|w|w||
|w|r|r||
|w|r|w||
|w|w|r||
|w|w|w||

### Just the possible ones.
www is not possible because we only have 2 white hats.

|C|D|E|Status|
|-|-|-|-|
|r|r|r||
|r|r|w||
|r|w|r||
|r|w|w||
|w|r|r||
|w|r|w||
|w|w|r||
|w|w|w|Not possible|

### Contestant C
If C sees 2 other white hats, she knows for sure that she has a red hat. So, that case can be resolved here, leaving 6 more possibilities.

|C|D|E|Status|
|-|-|-|-|
|r|r|r||
|r|r|w||
|r|w|r||
|r|w|w|C announces red|
|w|r|r||
|w|r|w||
|w|w|r||
|w|w|w|Not possible|

### Contestant D
If E has a white hat, then D knows that she has a red hat. Because, if not, C would have already guessed her hat color. So, we are down to 4 possibilities.

|C|D|E|Status|
|-|-|-|-|
|r|r|r||
|r|r|w|D announces red|
|r|w|r||
|r|w|w|C announces red|
|w|r|r||
|w|r|w|D announces red|
|w|w|r||
|w|w|w|Not possible|

### Contestant E
Now the remaining cases (rrr, rwr, wrr, wwr) all have E wearing a red hat.
So, if the game comes to E, she can confidently announce red.

|C|D|E|Status|
|-|-|-|-|
|r|r|r|E announces red|
|r|r|w|D announces red|
|r|w|r|E announces red|
|r|w|w|C announces red|
|w|r|r|E announces red|
|w|r|w|D announces red|
|w|w|r|E announces red|
|w|w|w|Not possible|

### Fiddler Conclusion
So, some contestant will always announce that they have a red hat, leading to victory for the team.

# Extra Credit Solution

I am going to try to automate the procedure above, but it seems quite tricky.

In [53]:
import itertools
from math import prod
def all_cap_assignment_cases_slow(R,W,B,num_players=4):
    bag = ['r'] * R + ['w'] * W + ['b'] * B
    assignments = list(set(itertools.permutations(bag, num_players)))
    return assignments

def all_cap_assignment_cases(R,W,B,num_players=4):
    assignments = []
    for p in itertools.product('rwb', repeat=num_players):
        if p.count('r') <= R and p.count('w') <= W and p.count('b') <= B:
            assignments.append(p)
    return assignments

assert (set(all_cap_assignment_cases_slow(3, 2, 0, 3)) == set(all_cap_assignment_cases(3, 2, 0, 3)))

In [54]:
# Just to check the speed.
#all_cap_assignment_cases(10, 9, 0, 10)

In [55]:
# The players that could be used to predict the current player are all subsets of all the other players in the game.
def get_possible_predictors(player, num_players=4):
    other_players = [i for i in range(num_players) if i != player]
    #print(f"Player: {player} :: Other players: {other_players}")
    all_subsets = []
    for r in range(len(other_players) + 1):
        for combo in itertools.combinations(other_players, r):
            subset = []
            for c in combo:
                subset.append(c)
            all_subsets.append(subset)
    return all_subsets
#get_possible_predictors(1, 4)
    

In [56]:
def get_prediction(cases, player, num_players, predictor):
    p_dict = {}
    for c in cases:
        p_key = ''
        for i in range(num_players):
            if i in predictor:
                p_key += c[i]
            elif i == player:
                p_key += '_'
            else:
                p_key += '*'
        p_value = c[player]
        if p_key not in p_dict:
            p_dict[p_key] = set(p_value)
        else:
            p_dict[p_key].add(p_value)
    return p_dict

In [57]:
def filter_cases(cases, pattern, player, cap_color_prediction):
    filtered_cases = []
    for c in cases:
        match = True
        for i in range(len(pattern)):
            if pattern[i] != '*' and pattern[i] != '_' and pattern[i] != c[i]:
                match = False
                break
        if match:
            assert(c[player] == cap_color_prediction)
        if not match:
            filtered_cases.append(c)
    return filtered_cases

In [58]:
def simulate_player_1pass(possible_cases, player, num_players=4, details=True):
    working_cases = possible_cases
    predictors = get_possible_predictors(player, num_players)
    for predictor in predictors:
        prediction = get_prediction(working_cases, player, num_players, predictor)
        for pattern, cap_color_predictions in prediction.items():
            if len(cap_color_predictions) == 1:
                specific_cap_prediction = list(cap_color_predictions)[0]
                if details:
                    print(f"Player {player} can predict their cap color({specific_cap_prediction}) with predictor {predictor} and pattern {pattern}.")
                working_cases = filter_cases(working_cases, pattern, player, specific_cap_prediction)
                if len(working_cases) == 0:
                    return working_cases
    return working_cases

In [59]:
# A player can make multiple passes of predictions, and each pass can further narrow down the possible cases. 
# The loop continues until no new cases can be removed by the predictions.
def simulate_player(possible_cases, player, num_players=4, details=True):
    start_len = len(possible_cases)
    while True:
        working_cases = simulate_player_1pass(possible_cases, player, num_players, details)
        new_len = len(working_cases)
        if new_len == start_len or new_len == 0:
            break
        else:
            possible_cases = working_cases
            start_len = len(possible_cases)
    return working_cases

In [60]:
def analyze_RWB_combo(R, W, B, num_players=4, details=True):
    cases = all_cap_assignment_cases(R, W, B, num_players)
    if details:
        print(f"\nNumber of initial cases for R={R}, W={W}, B={B}: {len(cases)}")
    for player in range(num_players):
        cases = simulate_player(cases, player, num_players, details)
        if details:
            print(f"After player {player}, number of remaining cases: {len(cases)}")
    if details:
        print(f"Remaining cases after all players have predicted: {len(cases)}")
    if len(cases) == 0:
        if details:
            print(f"No cases remain for R={R}, W={W}, B={B}. Team Wins!")
        return True
    else :
        if details:
            print(f"Cases remain for R={R}, W={W}, B={B}. Team Loses.")
        return False

In [61]:
analyze_RWB_combo(3, 2, 0, 3)


Number of initial cases for R=3, W=2, B=0: 7
Player 0 can predict their cap color(r) with predictor [1, 2] and pattern _ww.
After player 0, number of remaining cases: 6
Player 1 can predict their cap color(r) with predictor [2] and pattern *_w.
After player 1, number of remaining cases: 4
Player 2 can predict their cap color(r) with predictor [] and pattern **_.
After player 2, number of remaining cases: 0
Remaining cases after all players have predicted: 0
No cases remain for R=3, W=2, B=0. Team Wins!


True

Okay, that was some complex code, but it seems to be working correctly, at least for the fiddler case. Let's try going further.

In [62]:
winning_combos = {}
max_RWB_sum = -1
for R in range(5):
    for W in range(5):
        for B in range(5):
            RWB_sum = R + W + B            
            success = analyze_RWB_combo(R, W, B, 4, details=False)
            if success:
                if RWB_sum not in winning_combos:
                    winning_combos[RWB_sum] = []
                winning_combos[RWB_sum].append((R, W, B))
            if success and RWB_sum >= max_RWB_sum:
                max_RWB_sum = RWB_sum
                print(f"New max RWB sum combo found: {max_RWB_sum} for R={R}, W={W}, B={B}")


New max RWB sum combo found: 0 for R=0, W=0, B=0
New max RWB sum combo found: 1 for R=0, W=0, B=1
New max RWB sum combo found: 2 for R=0, W=0, B=2
New max RWB sum combo found: 3 for R=0, W=0, B=3
New max RWB sum combo found: 4 for R=0, W=0, B=4
New max RWB sum combo found: 4 for R=0, W=1, B=3
New max RWB sum combo found: 5 for R=0, W=1, B=4
New max RWB sum combo found: 5 for R=0, W=2, B=3
New max RWB sum combo found: 6 for R=0, W=2, B=4
New max RWB sum combo found: 6 for R=0, W=3, B=3
New max RWB sum combo found: 7 for R=0, W=3, B=4
New max RWB sum combo found: 7 for R=0, W=4, B=3
New max RWB sum combo found: 7 for R=1, W=2, B=4
New max RWB sum combo found: 7 for R=1, W=4, B=2
New max RWB sum combo found: 7 for R=2, W=1, B=4
New max RWB sum combo found: 7 for R=2, W=4, B=1
New max RWB sum combo found: 7 for R=3, W=0, B=4
New max RWB sum combo found: 7 for R=3, W=4, B=0
New max RWB sum combo found: 7 for R=4, W=0, B=3
New max RWB sum combo found: 7 for R=4, W=1, B=2
New max RWB sum comb

In [63]:
print(f"Maximum RWB sum for winning combinations: {max_RWB_sum}")
print(f"Winning combinations with max sum: {winning_combos[max_RWB_sum]}")

Maximum RWB sum for winning combinations: 7
Winning combinations with max sum: [(0, 3, 4), (0, 4, 3), (1, 2, 4), (1, 4, 2), (2, 1, 4), (2, 4, 1), (3, 0, 4), (3, 4, 0), (4, 0, 3), (4, 1, 2), (4, 2, 1), (4, 3, 0)]


Permutation apart, there seem to be 2 ways to get the the max RWB sum of 7 and still succeed. Let's check those in more detail.

In [64]:
analyze_RWB_combo(4, 3, 0, 4)


Number of initial cases for R=4, W=3, B=0: 15
Player 0 can predict their cap color(r) with predictor [1, 2, 3] and pattern _www.
After player 0, number of remaining cases: 14
Player 1 can predict their cap color(r) with predictor [2, 3] and pattern *_ww.
After player 1, number of remaining cases: 12
Player 2 can predict their cap color(r) with predictor [3] and pattern **_w.
After player 2, number of remaining cases: 8
Player 3 can predict their cap color(r) with predictor [] and pattern ***_.
After player 3, number of remaining cases: 0
Remaining cases after all players have predicted: 0
No cases remain for R=4, W=3, B=0. Team Wins!


True

In [65]:
analyze_RWB_combo(4, 2, 1, 4)


Number of initial cases for R=4, W=2, B=1: 39
Player 0 can predict their cap color(r) with predictor [1, 2, 3] and pattern _wwb.
Player 0 can predict their cap color(r) with predictor [1, 2, 3] and pattern _wbw.
Player 0 can predict their cap color(r) with predictor [1, 2, 3] and pattern _bww.
After player 0, number of remaining cases: 36
Player 1 can predict their cap color(r) with predictor [2, 3] and pattern *_ww.
Player 1 can predict their cap color(r) with predictor [2, 3] and pattern *_wb.
Player 1 can predict their cap color(r) with predictor [2, 3] and pattern *_bw.
After player 1, number of remaining cases: 30
Player 2 can predict their cap color(r) with predictor [3] and pattern **_w.
Player 2 can predict their cap color(r) with predictor [3] and pattern **_b.
After player 2, number of remaining cases: 19
Player 3 can predict their cap color(r) with predictor [] and pattern ***_.
After player 3, number of remaining cases: 0
Remaining cases after all players have predicted: 0

True

Looks good. Makes sense. And works similarly to the Fiddler case.

We can speculate that the pattern (N, N-1-k, k, N) continues to work for larger N. Let's check.

In [66]:
import time
non_success_count = 0
for N in range(1,10):
    for k in range(N):
        start_time = time.time()
        success = analyze_RWB_combo(N, N-1-k, k, N, details=False)
        end_time = time.time()
        print(f"R={N}, W={N-1-k}, B={k}, N={N} :: Success: {success} :: Time: {end_time - start_time:.2f} seconds")
        if not success:
            non_success_count += 1

print(f"Total non-successful cases: {non_success_count}")

R=1, W=0, B=0, N=1 :: Success: True :: Time: 0.00 seconds
R=2, W=1, B=0, N=2 :: Success: True :: Time: 0.00 seconds
R=2, W=0, B=1, N=2 :: Success: True :: Time: 0.00 seconds
R=3, W=2, B=0, N=3 :: Success: True :: Time: 0.00 seconds
R=3, W=1, B=1, N=3 :: Success: True :: Time: 0.00 seconds
R=3, W=0, B=2, N=3 :: Success: True :: Time: 0.00 seconds
R=4, W=3, B=0, N=4 :: Success: True :: Time: 0.00 seconds
R=4, W=2, B=1, N=4 :: Success: True :: Time: 0.00 seconds
R=4, W=1, B=2, N=4 :: Success: True :: Time: 0.00 seconds
R=4, W=0, B=3, N=4 :: Success: True :: Time: 0.00 seconds
R=5, W=4, B=0, N=5 :: Success: True :: Time: 0.00 seconds
R=5, W=3, B=1, N=5 :: Success: True :: Time: 0.01 seconds
R=5, W=2, B=2, N=5 :: Success: True :: Time: 0.02 seconds
R=5, W=1, B=3, N=5 :: Success: True :: Time: 0.01 seconds
R=5, W=0, B=4, N=5 :: Success: True :: Time: 0.00 seconds
R=6, W=5, B=0, N=6 :: Success: True :: Time: 0.01 seconds
R=6, W=4, B=1, N=6 :: Success: True :: Time: 0.04 seconds
R=6, W=3, B=2,

### Extra Credit Conclusion

The max RWB sum for the extra-credit is 7, reachable as (R,W,B) = (4,3,0) or (4,2,1) or permutation thereof.

The pattern of (N,N-1-k,k) having a successful strategy for the team continues till N=9 at least, and presumably ad infinitum thereafter.